# Cross-task failure memory on HotpotQA (DS 340W, Group 48)

Extends Reflexion (Shinn et al., 2023). The agent turns failures on **training** questions into lessons
(failure type + diagnosis + general fix), keeps only lessons that fix the task they came from, and
retrieves them **before the first attempt** on unseen **test** questions.

Conditions: `no_memory` (control), `reflexion` (parent paper), `failure_memory` (ours), `random_memory` (sanity check: same number of lessons, chosen at random).

**Setup:** add your Groq key in Colab Secrets (key icon on the left) as `GROQ_API_KEY`.

In [ ]:
!git clone -q https://github.com/omaraljashmi/reflexion-hotpotqa-reproduction.git
%cd reflexion-hotpotqa-reproduction
!pip -q install -r requirements.txt

In [ ]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
MODEL = "openai/gpt-oss-20b"   # try a different Groq model if the pilot is outside 40-75%

## 0. Offline tests (no API calls)

In [ ]:
!python -m unittest discover -s tests 2>&1 | tail -3

## 1. Pilot: is the model making enough mistakes to learn from?
Target first-attempt exact match: roughly 40-75%.

In [ ]:
!python run_experiment.py pilot --provider groq --model $MODEL --n 30

## 2. Build the lesson memory from training questions

In [ ]:
!python run_experiment.py build --provider groq --model $MODEL --n 150

In [ ]:
import json
lessons = json.load(open("results/memory.json"))["lessons"]
for l in [l for l in lessons if l["validated"]][:10]:
    print(f'{l["id"]} [{l["qtype"]}/{l["failure_type"]}]\n  diagnosis: {l["diagnosis"]}\n  fix: {l["fix"]}\n')

## 3. Evaluate on unseen test questions

In [ ]:
!python run_experiment.py evaluate --provider groq --model $MODEL --n 100 --trials 2

## 4. Analysis (reruns from the saved CSV, no API key needed)

In [ ]:
import pandas as pd
df = pd.read_csv("results/eval_attempts.csv")
first = df[df.trial == 1]
display(first.groupby("condition").exact_match.mean().rename("first-attempt EM"))
display(first[first.exact_match == 0].groupby(["condition", "failure_type"]).size().unstack(fill_value=0))

In [ ]:
!python run_experiment.py analyze --attempts results/eval_attempts.csv